# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from experiments.models import (
    DeepARForecast,
    TFTForecast,
    ChronosForecast,
)

from experiments.utils import (
    load_experiments_specs,
    create_tsfeatures,
)

# Load Experiment Specifications

In [ ]:
dataset = "ABC"

In [ ]:
# Load specifications
train_type = "global"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
dl_params = config["deep_learning"]

# TS-Features

In [ ]:
# Extract time series features
ts_feats_df, ts_feats = create_tsfeatures(
    train=train_df,
    freq=freq
)

# Add features to train and test
train_df = pd.merge(train_df, ts_feats_df, on="series_id", how="left")
test_df = pd.merge(test_df, ts_feats_df, on="series_id", how="left")
features = features + ts_feats

# DeepAR

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)
pl.seed_everything(seed=seed, workers=True)

deepar_fcst = DeepARForecast(
    train_df,
    test_df,
    meta,
    freq,
    fcst_h,
    lags,
    max_lag,
    config,
    series_ids,
    device
)

# TFT

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)
pl.seed_everything(seed=seed, workers=True)

tft_fcst = TFTForecast(
    train_df,
    test_df,
    meta,
    freq,
    fcst_h,
    config,
    series_ids,
    device
)

# Chronos

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)
pl.seed_everything(seed=seed, workers=True)

chronos_fcst = ChronosForecast(
    train_df,
    test_df.drop(columns=["value"]),
    series_ids,
    fcst_h,
    config,
    device
)

# Combine Forecasts

In [ ]:
# Concatenate all forecasts into a single DataFrame
fcsts_df = pd.concat(
    [
        deepar_fcst,
        tft_fcst,
        chronos_fcst,
    ],
    axis=0
)

# Add actual values and dataset information
fcsts_df = pd.merge(
    fcsts_df,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Save Forecasts

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / train_type
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_deeplearning_fcsts.csv", index=False)